In [1]:
from tqdm import tqdm

# Load database
try:
    curve_db
    print(f"✓ curve_db loaded with {len(curve_db)} isogeny classes")
except NameError:
    print("Loading curve database...")
    curve_db = load('Curve Database (Conductor < 100000)/curve_database_N1_to_100000.sobj')
    print(f"✓ Loaded curve_db with {len(curve_db)} isogeny classes")

# Build subdatabase of fixed curves
print('Building rank-restricted databases...')

rk0_curves = {}
rk1_curves = {}
rk2_curves = {}
rk3_curves = {}
for iso in tqdm(list(curve_db.keys()), 'Building databases'):
    rk = curve_db[iso]['rank']
    if rk == 0:
        rk0_curves[iso] = curve_db[iso]
    elif rk == 1:
        rk1_curves[iso] = curve_db[iso]
    elif rk == 2:
        rk2_curves[iso] = curve_db[iso]
    elif rk == 3:
        rk3_curves[iso] = curve_db[iso]

Loading curve database...
✓ Loaded curve_db with 437226 isogeny classes
Building rank-restricted databases...


Building databases: 100%|███████████████████████████████████████████████████| 437226/437226 [00:00<00:00, 749833.49it/s]


In [8]:
Nmin = 14000
Nmax = 15000

# ── Filter both ranks by conductor ──
Nisos_rk0, Ncurves_rk0 = [], {}
for iso in tqdm(list(rk0_curves.keys()), 'Conductor range (rank 0)'):
    N = rk0_curves[iso]['conductor']
    if Nmin <= N <= Nmax:
        Nisos_rk0.append(iso)
        Ncurves_rk0[iso] = rk0_curves[iso]

Nisos_rk1, Ncurves_rk1 = [], {}
for iso in tqdm(list(rk1_curves.keys()), 'Conductor range (rank 1)'):
    N = rk1_curves[iso]['conductor']
    if Nmin <= N <= Nmax:
        Nisos_rk1.append(iso)
        Ncurves_rk1[iso] = rk1_curves[iso]

# ── Build running averages separately ──
from random import shuffle
import numpy as np

shuffled_rk0 = Nisos_rk0[:]
shuffle(shuffled_rk0)
aps_ev_rk0 = []
for n, iso in enumerate(tqdm(shuffled_rk0, 'Avg aps (rank 0)')):
    anaps = np.array(Ncurves_rk0[iso]['ap_list'])
    if n == 0:
        avg_aps = anaps
    else:
        avg_aps = (aps_ev_rk0[-1] * n + anaps) / (n + 1)
    aps_ev_rk0.append(avg_aps)

shuffled_rk1 = Nisos_rk1[:]
shuffle(shuffled_rk1)
aps_ev_rk1 = []
for n, iso in enumerate(tqdm(shuffled_rk1, 'Avg aps (rank 1)')):
    anaps = np.array(Ncurves_rk1[iso]['ap_list'])
    if n == 0:
        avg_aps = anaps
    else:
        avg_aps = (aps_ev_rk1[-1] * n + anaps) / (n + 1)
    aps_ev_rk1.append(avg_aps)

Avg aps (rank 1): 100%|███████████████████████████████████████████████████████████| 2217/2217 [00:00<00:00, 9322.12it/s]


In [9]:
"""
Animate the evolution of the average a_p vector for rank 0 and rank 1,
with murmuration fit  y = A * j^alpha * sin(B * j^beta)  over all primes,
and a live chi^2 evolution subplot.
Requires: aps_ev_rk0, aps_ev_rk1, Nmin, Nmax already computed.

Usage (in SageMath/Jupyter):
    %run animate_aps_2rank.py
"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib import rcParams
from scipy.optimize import curve_fit
from tqdm import tqdm
import os

# ── Style ────────────────────────────────────────────────────────────────
plt.style.use('dark_background')
rcParams['font.family'] = 'monospace'
rcParams['font.size'] = 11

# ── Data setup ───────────────────────────────────────────────────────────
aps_matrix_rk0 = np.array(aps_ev_rk0)
aps_matrix_rk1 = np.array(aps_ev_rk1)
n_curves_rk0, n_primes = aps_matrix_rk0.shape
n_curves_rk1 = len(aps_matrix_rk1)
n_steps = max(n_curves_rk0, n_curves_rk1)
x = np.arange(n_primes)

# j = 1..n_primes for fitting (avoid 0^alpha)
j_all = np.arange(1, n_primes + 1, dtype=float)
j_smooth = np.linspace(1, n_primes, 1000)

ymin = -6
ymax = 6

# ── Fit model ────────────────────────────────────────────────────────────
def murmuration(j, A, alpha, B, beta):
    return A * np.power(j, alpha) * np.sin(B * np.power(j, beta))

# Frame selection
max_frames = 300
if n_steps <= max_frames:
    frame_indices = list(range(n_steps))
else:
    frame_indices = sorted(set(
        list(np.linspace(0, n_steps - 1, max_frames, dtype=int))
    ))
n_frames = len(frame_indices)

# ── Precompute fits ──────────────────────────────────────────────────────
print('Precomputing murmuration fits...')

def fit_murmuration(aps_matrix, n_curves, sign=1.0):
    """Fit y = A j^alpha sin(B j^beta). Returns fits dict and chi2 list."""
    fits = {}
    chi2_list = []
    curve_counts = []
    unique_indices = sorted(set(min(idx, n_curves - 1) for idx in frame_indices))

    for i in tqdm(unique_indices, desc=f'Fitting (sign={sign:+.0f})'):
        y = aps_matrix[i]
        p0_list = [
            [sign * 1.0, 0.5, 0.5, 0.5],
            [sign * 2.0, 0.3, 0.3, 0.6],
            [sign * 0.5, 0.7, 0.8, 0.4],
        ]
        best_popt = None
        best_res = np.inf
        if sign > 0:
            bnds = ([0, 0, 0, 0], [np.inf, 2, 1.0, 2])
        else:
            bnds = ([-np.inf, 0, 0, 0], [0, 2, 1.0, 2])
        for p0 in p0_list:
            try:
                popt, _ = curve_fit(
                    murmuration, j_all, y, p0=p0, maxfev=10000,
                    bounds=bnds
                )
                residual = np.sum((y - murmuration(j_all, *popt))**2)
                if residual < best_res:
                    best_res = residual
                    best_popt = popt
            except Exception:
                continue

        if best_popt is not None:
            A, alpha, B, beta = best_popt
            y_curve = murmuration(j_smooth, *best_popt)
            chi2 = best_res / (n_primes - 4)  # reduced chi^2
        else:
            A, alpha, B, beta = [np.nan] * 4
            y_curve = np.full_like(j_smooth, np.nan)
            chi2 = np.nan

        fits[i] = (A, alpha, B, beta, y_curve, chi2)
        chi2_list.append(chi2)
        curve_counts.append(i + 1)

    return fits, np.array(curve_counts), np.array(chi2_list)

fits_rk0, counts_rk0, chi2_rk0 = fit_murmuration(aps_matrix_rk0, n_curves_rk0, sign=+1.0)
fits_rk1, counts_rk1, chi2_rk1 = fit_murmuration(aps_matrix_rk1, n_curves_rk1, sign=-1.0)
print('✓ Fits precomputed')

# ── Figure ───────────────────────────────────────────────────────────────
fig, (ax_main, ax_chi2) = plt.subplots(
    2, 1, figsize=(12, 7), dpi=120,
    gridspec_kw={'height_ratios': [3, 1], 'hspace': 0.3}
)
fig.set_facecolor('#0a0a0f')

# ── Main scatter axes ────────────────────────────────────────────────────
ax_main.set_facecolor('#0a0a0f')
ax_main.grid(True, alpha=0.12, color='#ffffff', linewidth=0.5)
ax_main.set_xlim(0, n_primes - 1)
ax_main.set_ylim(ymin, ymax)
ax_main.set_xlabel('Prime index $j$', fontsize=12, color='#aaaaaa', labelpad=8)
ax_main.set_ylabel(r'$\langle a_{p_j} \rangle$', fontsize=14, color='#aaaaaa', labelpad=8)
ax_main.tick_params(colors='#666666')
for spine in ax_main.spines.values():
    spine.set_color('#222233')
ax_main.axhline(0, color='#333344', linewidth=0.8, zorder=1)

scatter0 = ax_main.scatter([], [], s=1.5, c='#00e5ff', alpha=0.7, zorder=3, label='rank 0')
scatter1 = ax_main.scatter([], [], s=1.5, c='#ff6e6e', alpha=0.7, zorder=3, label='rank 1')
fit_line0, = ax_main.plot([], [], lw=2, color='#00e5ff', alpha=0.9, zorder=5)
fit_line1, = ax_main.plot([], [], lw=2, color='#ff6e6e', alpha=0.9, zorder=5)
ax_main.legend(loc='upper left', fontsize=10, framealpha=0.3, edgecolor='#333344')

title = ax_main.set_title('', fontsize=13, color='#e0e0e0', pad=12, loc='left')
counter = ax_main.text(0.98, 0.95, '', transform=ax_main.transAxes, fontsize=18,
                       fontweight='bold', color='#888888', alpha=0.7,
                       ha='right', va='top', fontfamily='monospace')

fit_text0 = ax_main.text(0.02, 0.08, '', transform=ax_main.transAxes, fontsize=9,
                         color='#00e5ff', alpha=0.8, fontfamily='monospace', va='bottom')
fit_text1 = ax_main.text(0.02, 0.02, '', transform=ax_main.transAxes, fontsize=9,
                         color='#ff6e6e', alpha=0.8, fontfamily='monospace', va='bottom')

ax_main.text(0.98, 0.06, f'N ∈ [{Nmin}, {Nmax}]',
             transform=ax_main.transAxes, fontsize=10, color='#555566',
             ha='right', va='bottom', fontfamily='monospace')

# ── Chi^2 axes ───────────────────────────────────────────────────────────
ax_chi2.set_facecolor('#0a0a0f')
ax_chi2.grid(True, alpha=0.12, color='#ffffff', linewidth=0.5)
ax_chi2.set_xlim(0, n_steps)
# Set y-range from precomputed data, ignoring early noisy frames
chi2_all = np.concatenate([chi2_rk0[~np.isnan(chi2_rk0)],
                           chi2_rk1[~np.isnan(chi2_rk1)]])
if len(chi2_all) > 0:
    chi2_ymax = np.percentile(chi2_all, 95) * 1.3
    chi2_ymin = max(0, min(chi2_all) * 0.5)
else:
    chi2_ymax = 10
    chi2_ymin = 0
ax_chi2.set_ylim(chi2_ymin, chi2_ymax)
ax_chi2.set_xlabel('Curves added', fontsize=11, color='#aaaaaa', labelpad=6)
ax_chi2.set_ylabel(r'$\chi^2_\nu$', fontsize=12, color='#aaaaaa', labelpad=8)
ax_chi2.tick_params(colors='#666666')
for spine in ax_chi2.spines.values():
    spine.set_color('#222233')

chi2_line0, = ax_chi2.plot([], [], lw=1.5, color='#00e5ff', alpha=0.85, label='rank 0')
chi2_line1, = ax_chi2.plot([], [], lw=1.5, color='#ff6e6e', alpha=0.85, label='rank 1')
chi2_dot0, = ax_chi2.plot([], [], 'o', color='#00e5ff', markersize=5, alpha=0.9, zorder=5)
chi2_dot1, = ax_chi2.plot([], [], 'o', color='#ff6e6e', markersize=5, alpha=0.9, zorder=5)
ax_chi2.legend(loc='upper right', fontsize=9, framealpha=0.3, edgecolor='#333344')

fig.subplots_adjust(top=0.92, bottom=0.08)

# ── Build frame-indexed chi2 arrays ──────────────────────────────────────
# Map from frame index to position in the chi2 arrays
frame_chi2_x_rk0, frame_chi2_y_rk0 = [], []
frame_chi2_x_rk1, frame_chi2_y_rk1 = [], []

for frame in range(n_frames):
    idx = frame_indices[frame]

    idx0 = min(idx, n_curves_rk0 - 1)
    _, _, _, _, _, c0 = fits_rk0[idx0]
    frame_chi2_x_rk0.append(idx0 + 1)
    frame_chi2_y_rk0.append(c0)

    idx1 = min(idx, n_curves_rk1 - 1)
    _, _, _, _, _, c1 = fits_rk1[idx1]
    frame_chi2_x_rk1.append(idx1 + 1)
    frame_chi2_y_rk1.append(c1)

# ── Animation ────────────────────────────────────────────────────────────
def init():
    scatter0.set_offsets(np.empty((0, 2)))
    scatter1.set_offsets(np.empty((0, 2)))
    fit_line0.set_data([], [])
    fit_line1.set_data([], [])
    chi2_line0.set_data([], [])
    chi2_line1.set_data([], [])
    chi2_dot0.set_data([], [])
    chi2_dot1.set_data([], [])
    title.set_text('')
    counter.set_text('')
    fit_text0.set_text('')
    fit_text1.set_text('')
    return (scatter0, scatter1, fit_line0, fit_line1,
            chi2_line0, chi2_line1, chi2_dot0, chi2_dot1,
            title, counter, fit_text0, fit_text1)

def update(frame):
    idx = frame_indices[frame]

    # ── Main plot ──
    idx0 = min(idx, n_curves_rk0 - 1)
    y0 = aps_matrix_rk0[idx0]
    scatter0.set_offsets(np.column_stack([x, y0]))
    A0, alpha0, B0, beta0, yc0, c0 = fits_rk0[idx0]
    fit_line0.set_data(j_smooth - 1, yc0)
    if not np.isnan(A0):
        fit_text0.set_text(
            f'rk0: {A0:+.2f} · j^{alpha0:.2f} · sin({B0:.2f} · j^{beta0:.2f})'
        )
    else:
        fit_text0.set_text('rk0: fit failed')

    idx1 = min(idx, n_curves_rk1 - 1)
    y1 = aps_matrix_rk1[idx1]
    scatter1.set_offsets(np.column_stack([x, y1]))
    A1, alpha1, B1, beta1, yc1, c1 = fits_rk1[idx1]
    fit_line1.set_data(j_smooth - 1, yc1)
    if not np.isnan(A1):
        fit_text1.set_text(
            f'rk1: {A1:+.2f} · j^{alpha1:.2f} · sin({B1:.2f} · j^{beta1:.2f})'
        )
    else:
        fit_text1.set_text('rk1: fit failed')

    n0, n1 = idx0 + 1, idx1 + 1
    title.set_text(
        f'Murmuration:  rank 0 ({n0}/{n_curves_rk0})  ·  rank 1 ({n1}/{n_curves_rk1})'
    )
    counter.set_text(f'{min(idx + 1, n_steps)}/{n_steps}')

    # ── Chi^2 plot ──
    cx0 = frame_chi2_x_rk0[:frame + 1]
    cy0 = frame_chi2_y_rk0[:frame + 1]
    chi2_line0.set_data(cx0, cy0)
    chi2_dot0.set_data([cx0[-1]], [cy0[-1]])

    cx1 = frame_chi2_x_rk1[:frame + 1]
    cy1 = frame_chi2_y_rk1[:frame + 1]
    chi2_line1.set_data(cx1, cy1)
    chi2_dot1.set_data([cx1[-1]], [cy1[-1]])

    return (scatter0, scatter1, fit_line0, fit_line1,
            chi2_line0, chi2_line1, chi2_dot0, chi2_dot1,
            title, counter, fit_text0, fit_text1)

anim = animation.FuncAnimation(
    fig, update, frames=n_frames,
    init_func=init, blit=True, interval=40
)

# ── Save ─────────────────────────────────────────────────────────────────
os.makedirs('MurmurationFormation', exist_ok=True)
outpath = f'MurmurationFormation/aps_evolution_rank01_N{Nmin}_{Nmax}.mp4'
try:
    writer = animation.FFMpegWriter(fps=25, metadata={'title': 'a_p murmuration (rank 0+1)'})
    anim.save(outpath, writer=writer)
    print(f'✓ Saved animation to {outpath}')
except Exception as e:
    print(f'FFmpeg not available ({e}), trying GIF...')
    outpath = f'MurmurationFormation/aps_evolution_rank01_N{Nmin}_{Nmax}.gif'
    writer = animation.PillowWriter(fps=20)
    anim.save(outpath, writer=writer)
    print(f'✓ Saved animation to {outpath}')

plt.close(fig)

Precomputing murmuration fits...


Fitting (sign=-1): 100%|██████████████████████████████████████████████████████████████| 300/300 [01:01<00:00,  4.87it/s]


✓ Fits precomputed
✓ Saved animation to MurmurationFormation/aps_evolution_rank01_N14000_15000.mp4


In [13]:
"""
Animate the evolution of the average a_p vector for rank 0 and rank 1,
with murmuration fit  y = A * j^alpha * sin(B * j^beta)  over all primes,
and a live chi^2 evolution subplot.
Requires: aps_ev_rk0, aps_ev_rk1, Nmin, Nmax already computed.

Usage (in SageMath/Jupyter):
    %run animate_aps_2rank.py
"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib import rcParams
from scipy.optimize import curve_fit
from tqdm import tqdm
import os

# ── Style ────────────────────────────────────────────────────────────────
plt.style.use('dark_background')
rcParams['font.family'] = 'monospace'
rcParams['font.size'] = 11

# ── Data setup ───────────────────────────────────────────────────────────
aps_matrix_rk0 = np.array(aps_ev_rk0)
aps_matrix_rk1 = np.array(aps_ev_rk1)
n_curves_rk0, n_primes = aps_matrix_rk0.shape
n_curves_rk1 = len(aps_matrix_rk1)
n_steps = max(n_curves_rk0, n_curves_rk1)
x = np.arange(n_primes)

# j = 1..n_primes for fitting (avoid 0^alpha)
j_all = np.arange(1, n_primes + 1, dtype=float)
j_smooth = np.linspace(1, n_primes, 1000)

ymin = -6
ymax = 6

# ── Fit model ────────────────────────────────────────────────────────────
# ── Tunable fit bounds ───────────────────────────────────────────────────
B_max = 1.0  # upper bound on B; increase if fit undershoots oscillation frequency

def murmuration(j, A, alpha, B, beta):
    return A * np.power(j, alpha) * np.sin(B * np.power(j, beta))

# Frame selection
max_frames = 300
if n_steps <= max_frames:
    frame_indices = list(range(n_steps))
else:
    frame_indices = sorted(set(
        list(np.linspace(0, n_steps - 1, max_frames, dtype=int))
    ))
n_frames = len(frame_indices)

# ── Precompute fits ──────────────────────────────────────────────────────
print('Precomputing murmuration fits...')

def fit_murmuration(aps_matrix, n_curves, sign=1.0):
    """Fit y = A j^alpha sin(B j^beta). Returns fits dict and chi2 list."""
    fits = {}
    chi2_list = []
    curve_counts = []
    unique_indices = sorted(set(min(idx, n_curves - 1) for idx in frame_indices))

    for i in tqdm(unique_indices, desc=f'Fitting (sign={sign:+.0f})'):
        y = aps_matrix[i]
        p0_list = [
            [sign * 1.0, 0.5, 0.5, 0.5],
            [sign * 2.0, 0.3, 0.3, 0.6],
            [sign * 0.5, 0.7, 0.8, 0.4],
        ]
        best_popt = None
        best_res = np.inf
        if sign > 0:
            bnds = ([0, 0, 0, 0], [np.inf, 2, B_max, 2])
        else:
            bnds = ([-np.inf, 0, 0, 0], [0, 2, B_max, 2])
        for p0 in p0_list:
            try:
                popt, _ = curve_fit(
                    murmuration, j_all, y, p0=p0, maxfev=10000,
                    bounds=bnds
                )
                residual = np.sum((y - murmuration(j_all, *popt))**2)
                if residual < best_res:
                    best_res = residual
                    best_popt = popt
            except Exception:
                continue

        if best_popt is not None:
            A, alpha, B, beta = best_popt
            y_curve = murmuration(j_smooth, *best_popt)
            chi2 = best_res / (n_primes - 4)  # reduced chi^2
        else:
            A, alpha, B, beta = [np.nan] * 4
            y_curve = np.full_like(j_smooth, np.nan)
            chi2 = np.nan

        fits[i] = (A, alpha, B, beta, y_curve, chi2)
        chi2_list.append(chi2)
        curve_counts.append(i + 1)

    return fits, np.array(curve_counts), np.array(chi2_list)

fits_rk0, counts_rk0, chi2_rk0 = fit_murmuration(aps_matrix_rk0, n_curves_rk0, sign=+1.0)
fits_rk1, counts_rk1, chi2_rk1 = fit_murmuration(aps_matrix_rk1, n_curves_rk1, sign=-1.0)
print('✓ Fits precomputed')

# ── Figure ───────────────────────────────────────────────────────────────
fig, (ax_main, ax_chi2) = plt.subplots(
    2, 1, figsize=(12, 7), dpi=120,
    gridspec_kw={'height_ratios': [3, 1], 'hspace': 0.3}
)
fig.set_facecolor('#0a0a0f')

# ── Main scatter axes ────────────────────────────────────────────────────
ax_main.set_facecolor('#0a0a0f')
ax_main.grid(True, alpha=0.12, color='#ffffff', linewidth=0.5)
ax_main.set_xlim(0, n_primes - 1)
ax_main.set_ylim(ymin, ymax)
ax_main.set_xlabel('Prime index $j$', fontsize=12, color='#aaaaaa', labelpad=8)
ax_main.set_ylabel(r'$\langle a_{p_j} \rangle$', fontsize=14, color='#aaaaaa', labelpad=8)
ax_main.tick_params(colors='#666666')
for spine in ax_main.spines.values():
    spine.set_color('#222233')
ax_main.axhline(0, color='#333344', linewidth=0.8, zorder=1)

scatter0 = ax_main.scatter([], [], s=1.5, c='#00e5ff', alpha=0.7, zorder=3, label='rank 0')
scatter1 = ax_main.scatter([], [], s=1.5, c='#ff6e6e', alpha=0.7, zorder=3, label='rank 1')
fit_line0, = ax_main.plot([], [], lw=2, color='#00e5ff', alpha=0.9, zorder=5)
fit_line1, = ax_main.plot([], [], lw=2, color='#ff6e6e', alpha=0.9, zorder=5)
ax_main.legend(loc='upper left', fontsize=10, framealpha=0.3, edgecolor='#333344')

title = ax_main.set_title('', fontsize=13, color='#e0e0e0', pad=12, loc='left')
counter = ax_main.text(0.98, 0.95, '', transform=ax_main.transAxes, fontsize=18,
                       fontweight='bold', color='#888888', alpha=0.7,
                       ha='right', va='top', fontfamily='monospace')

fit_text0 = ax_main.text(0.02, 0.08, '', transform=ax_main.transAxes, fontsize=9,
                         color='#00e5ff', alpha=0.8, fontfamily='monospace', va='bottom')
fit_text1 = ax_main.text(0.02, 0.02, '', transform=ax_main.transAxes, fontsize=9,
                         color='#ff6e6e', alpha=0.8, fontfamily='monospace', va='bottom')

ax_main.text(0.98, 0.06, f'N ∈ [{Nmin}, {Nmax}]',
             transform=ax_main.transAxes, fontsize=10, color='#555566',
             ha='right', va='bottom', fontfamily='monospace')

# ── Chi^2 axes ───────────────────────────────────────────────────────────
ax_chi2.set_facecolor('#0a0a0f')
ax_chi2.grid(True, alpha=0.12, color='#ffffff', linewidth=0.5)
ax_chi2.set_xlim(0, n_steps)
# Set y-range from precomputed data, ignoring early noisy frames
chi2_all = np.concatenate([chi2_rk0[~np.isnan(chi2_rk0)],
                           chi2_rk1[~np.isnan(chi2_rk1)]])
if len(chi2_all) > 0:
    chi2_ymax = np.percentile(chi2_all, 95) * 1.3
    chi2_ymin = max(0, min(chi2_all) * 0.5)
else:
    chi2_ymax = 10
    chi2_ymin = 0
ax_chi2.set_ylim(chi2_ymin, chi2_ymax)
ax_chi2.set_xlabel('Curves added', fontsize=11, color='#aaaaaa', labelpad=6)
ax_chi2.set_ylabel(r'$\chi^2_\nu$', fontsize=12, color='#aaaaaa', labelpad=8)
ax_chi2.tick_params(colors='#666666')
for spine in ax_chi2.spines.values():
    spine.set_color('#222233')

chi2_line0, = ax_chi2.plot([], [], lw=1.5, color='#00e5ff', alpha=0.85, label='rank 0')
chi2_line1, = ax_chi2.plot([], [], lw=1.5, color='#ff6e6e', alpha=0.85, label='rank 1')
chi2_dot0, = ax_chi2.plot([], [], 'o', color='#00e5ff', markersize=5, alpha=0.9, zorder=5)
chi2_dot1, = ax_chi2.plot([], [], 'o', color='#ff6e6e', markersize=5, alpha=0.9, zorder=5)
ax_chi2.legend(loc='upper right', fontsize=9, framealpha=0.3, edgecolor='#333344')

fig.subplots_adjust(top=0.92, bottom=0.08)

# ── Build frame-indexed chi2 arrays ──────────────────────────────────────
# Map from frame index to position in the chi2 arrays
frame_chi2_x_rk0, frame_chi2_y_rk0 = [], []
frame_chi2_x_rk1, frame_chi2_y_rk1 = [], []

for frame in range(n_frames):
    idx = frame_indices[frame]

    idx0 = min(idx, n_curves_rk0 - 1)
    _, _, _, _, _, c0 = fits_rk0[idx0]
    frame_chi2_x_rk0.append(idx0 + 1)
    frame_chi2_y_rk0.append(c0)

    idx1 = min(idx, n_curves_rk1 - 1)
    _, _, _, _, _, c1 = fits_rk1[idx1]
    frame_chi2_x_rk1.append(idx1 + 1)
    frame_chi2_y_rk1.append(c1)

# ── Animation ────────────────────────────────────────────────────────────
def init():
    scatter0.set_offsets(np.empty((0, 2)))
    scatter1.set_offsets(np.empty((0, 2)))
    fit_line0.set_data([], [])
    fit_line1.set_data([], [])
    chi2_line0.set_data([], [])
    chi2_line1.set_data([], [])
    chi2_dot0.set_data([], [])
    chi2_dot1.set_data([], [])
    title.set_text('')
    counter.set_text('')
    fit_text0.set_text('')
    fit_text1.set_text('')
    return (scatter0, scatter1, fit_line0, fit_line1,
            chi2_line0, chi2_line1, chi2_dot0, chi2_dot1,
            title, counter, fit_text0, fit_text1)

def update(frame):
    idx = frame_indices[frame]

    # ── Main plot ──
    idx0 = min(idx, n_curves_rk0 - 1)
    y0 = aps_matrix_rk0[idx0]
    scatter0.set_offsets(np.column_stack([x, y0]))
    A0, alpha0, B0, beta0, yc0, c0 = fits_rk0[idx0]
    fit_line0.set_data(j_smooth - 1, yc0)
    if not np.isnan(A0):
        fit_text0.set_text(
            f'rk0: {A0:+.2f} · j^{alpha0:.2f} · sin({B0:.2f} · j^{beta0:.2f})'
        )
    else:
        fit_text0.set_text('rk0: fit failed')

    idx1 = min(idx, n_curves_rk1 - 1)
    y1 = aps_matrix_rk1[idx1]
    scatter1.set_offsets(np.column_stack([x, y1]))
    A1, alpha1, B1, beta1, yc1, c1 = fits_rk1[idx1]
    fit_line1.set_data(j_smooth - 1, yc1)
    if not np.isnan(A1):
        fit_text1.set_text(
            f'rk1: {A1:+.2f} · j^{alpha1:.2f} · sin({B1:.2f} · j^{beta1:.2f})'
        )
    else:
        fit_text1.set_text('rk1: fit failed')

    n0, n1 = idx0 + 1, idx1 + 1
    title.set_text(
        f'Murmuration:  rank 0 ({n0}/{n_curves_rk0})  ·  rank 1 ({n1}/{n_curves_rk1})'
    )
    counter.set_text(f'{min(idx + 1, n_steps)}/{n_steps}')

    # ── Chi^2 plot ──
    cx0 = frame_chi2_x_rk0[:frame + 1]
    cy0 = frame_chi2_y_rk0[:frame + 1]
    chi2_line0.set_data(cx0, cy0)
    chi2_dot0.set_data([cx0[-1]], [cy0[-1]])

    cx1 = frame_chi2_x_rk1[:frame + 1]
    cy1 = frame_chi2_y_rk1[:frame + 1]
    chi2_line1.set_data(cx1, cy1)
    chi2_dot1.set_data([cx1[-1]], [cy1[-1]])

    return (scatter0, scatter1, fit_line0, fit_line1,
            chi2_line0, chi2_line1, chi2_dot0, chi2_dot1,
            title, counter, fit_text0, fit_text1)

anim = animation.FuncAnimation(
    fig, update, frames=n_frames,
    init_func=init, blit=True, interval=40
)

# ── Save ─────────────────────────────────────────────────────────────────
os.makedirs('MurmurationFormation', exist_ok=True)
outpath = f'MurmurationFormation/aps_evolution_rank01_N{Nmin}_{Nmax}.mp4'
try:
    writer = animation.FFMpegWriter(fps=25, metadata={'title': 'a_p murmuration (rank 0+1)'})
    anim.save(outpath, writer=writer)
    print(f'✓ Saved animation to {outpath}')
except Exception as e:
    print(f'FFmpeg not available ({e}), trying GIF...')
    outpath = f'MurmurationFormation/aps_evolution_rank01_N{Nmin}_{Nmax}.gif'
    writer = animation.PillowWriter(fps=20)
    anim.save(outpath, writer=writer)
    print(f'✓ Saved animation to {outpath}')

plt.close(fig)

Precomputing murmuration fits...


Fitting (sign=-1): 100%|██████████████████████████████████████████████████████████████| 300/300 [01:02<00:00,  4.83it/s]


✓ Fits precomputed
✓ Saved animation to MurmurationFormation/aps_evolution_rank01_N14000_15000.mp4
